In [ ]:
import os, sys, textwrap

# === Setup local: pacote pubensino embutido no notebook (sem Drive) ===
ROOT = '/content/pubensino/ht'
os.makedirs(ROOT, exist_ok=True)

def _write(path, content):
    with open(path, 'w', encoding='utf-8') as f:
        f.write(content)

# cria pacotes
_write('/content/pubensino/__init__.py', '')
_write(os.path.join(ROOT, '__init__.py'), textwrap.dedent('# Heat transfer subpackage (ht)\n'))
_write(os.path.join(ROOT, 'wall1d.py'), textwrap.dedent('# pubensino/ht/wall1d.py\n# Condução 1D em regime permanente em parede plana\n# Convenção: temperaturas em Kelvin (K)\n\nfrom dataclasses import dataclass\nfrom typing import Tuple\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass Wall1DParams:\n    k: float    # W/(m·K)\n    L: float    # m\n    A: float    # m²\n    T1: float   # K\n    T2: float   # K\n\n\ndef validate_params(p: Wall1DParams) -> None:\n    """Validações físicas mínimas."""\n    if p.k <= 0:\n        raise ValueError("k deve ser > 0 [W/(m·K)].")\n    if p.L <= 0:\n        raise ValueError("L deve ser > 0 [m].")\n    if p.A <= 0:\n        raise ValueError("A deve ser > 0 [m²].")\n    if p.T1 <= 0 or p.T2 <= 0:\n        raise ValueError("Temperaturas devem estar em Kelvin (T > 0 K).")\n\n\ndef solve_wall_steady_1d(p: Wall1DParams, nx: int = 200) -> Tuple[np.ndarray, np.ndarray, float, float]:\n    """\n    Solução analítica para parede plana 1D, regime permanente:\n      T(x) linear; q\'\' constante.\n    Retorna:\n      x [m], T(x) [K], q\'\' [W/m²], Qdot [W]\n    """\n    validate_params(p)\n    if nx < 10:\n        raise ValueError("nx deve ser >= 10.")\n\n    x = np.linspace(0.0, p.L, nx)\n    T = p.T1 + (p.T2 - p.T1) * (x / p.L)\n\n    dTdx = (p.T2 - p.T1) / p.L\n    qpp = -p.k * dTdx           # W/m²\n    Qdot = qpp * p.A            # W\n\n    return x, T, qpp, Qdot\n\n\ndef thermal_resistance(p: Wall1DParams) -> float:\n    """Resistência térmica por condução (parede plana): R_cond = L/(kA) [K/W]."""\n    validate_params(p)\n    return p.L / (p.k * p.A)\n\n\ndef check_consistency(p: Wall1DParams) -> float:\n    """\n    Retorna erro relativo entre:\n      Qdot (Fourier) vs Qdot via resistência térmica.\n    """\n    _, _, _, Qdot = solve_wall_steady_1d(p)\n    R = thermal_resistance(p)\n    Qdot_R = (p.T1 - p.T2) / R\n\n    rel_err = abs(Qdot_R - Qdot) / (abs(Qdot) + 1e-15)\n    return rel_err\n\n\ndef _sanity() -> str:\n    return "wall1d OK"\n\n\nfrom dataclasses import dataclass\nfrom typing import Tuple\nimport numpy as np\n\n@dataclass(frozen=True)\nclass Wall1DConvParams:\n    k: float    # W/(m·K)\n    L: float    # m\n    A: float    # m²\n    T1: float   # K\n    h: float    # W/(m²·K)\n    Tinf: float # K\n\ndef validate_params_conv(p: Wall1DConvParams) -> None:\n    if p.k <= 0:\n        raise ValueError("k deve ser > 0 [W/(m·K)].")\n    if p.L <= 0:\n        raise ValueError("L deve ser > 0 [m].")\n    if p.A <= 0:\n        raise ValueError("A deve ser > 0 [m²].")\n    if p.h <= 0:\n        raise ValueError("h deve ser > 0 [W/(m²·K)].")\n    if p.T1 <= 0 or p.Tinf <= 0:\n        raise ValueError("Temperaturas devem estar em Kelvin (T > 0 K).")\n\ndef solve_wall_steady_1d_conv(p: Wall1DConvParams, nx: int = 200) -> Tuple[np.ndarray, np.ndarray, float, float, float]:\n    """\n    Parede plana 1D, regime permanente:\n      T(0)=T1 e convecção em x=L: -k dT/dx = h (T(L)-Tinf)\n    Retorna:\n      x [m], T(x) [K], q\'\' [W/m²], Qdot [W], Ts=T(L) [K]\n    """\n    validate_params_conv(p)\n    if nx < 10:\n        raise ValueError("nx deve ser >= 10.")\n\n    # Fluxo fechado\n    qpp = (p.T1 - p.Tinf) / (p.L/p.k + 1.0/p.h)   # W/m²\n\n    # Perfil linear\n    x = np.linspace(0.0, p.L, nx)\n    T = p.T1 - (qpp/p.k) * x                      # K\n\n    Ts = T[-1]\n    Qdot = qpp * p.A\n    return x, T, qpp, Qdot, Ts\n\n\n# ============================\n# Condução radial 1D — Cilindro (regime permanente, sem geração)\n# ============================\n\nfrom dataclasses import dataclass\nfrom typing import Tuple\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass Cyl1DParams:\n    k: float     # W/(m·K)\n    ri: float    # m\n    ro: float    # m\n    L: float     # m (comprimento do cilindro)\n    Ti: float    # K (em r=ri)\n    To: float    # K (em r=ro)\n\n\ndef validate_params_cyl(p: Cyl1DParams) -> None:\n    if p.k <= 0:\n        raise ValueError("k deve ser > 0 [W/(m·K)].")\n    if p.L <= 0:\n        raise ValueError("L deve ser > 0 [m].")\n    if p.ri <= 0 or p.ro <= 0:\n        raise ValueError("ri e ro devem ser > 0 [m].")\n    if p.ro <= p.ri:\n        raise ValueError("Deve valer ro > ri.")\n    if p.Ti <= 0 or p.To <= 0:\n        raise ValueError("Temperaturas devem estar em Kelvin (T > 0 K).")\n\n\ndef solve_cylinder_steady_1d(p: Cyl1DParams, nr: int = 300) -> Tuple[np.ndarray, np.ndarray, float, np.ndarray]:\n    """\n    Casca cilíndrica (1D radial), regime permanente, k const, sem geração.\n    BCs: T(ri)=Ti, T(ro)=To.\n\n    Retorna:\n      r [m], T(r) [K], Qdot [W], qpp(r) [W/m²] onde qpp(r)=Qdot/(2π r L)\n    """\n    validate_params_cyl(p)\n    if nr < 20:\n        raise ValueError("nr deve ser >= 20.")\n\n    r = np.linspace(p.ri, p.ro, nr)\n\n    # Perfil de temperatura (logarítmico)\n    lnR = np.log(p.ro / p.ri)\n    T = p.Ti + (p.To - p.Ti) * (np.log(r / p.ri) / lnR)\n\n    # Taxa de calor (constante)\n    Qdot = 2.0 * np.pi * p.L * p.k * (p.Ti - p.To) / lnR\n\n    # Fluxo radial por área local\n    qpp = Qdot / (2.0 * np.pi * r * p.L)\n\n    return r, T, Qdot, qpp\n\n\ndef thermal_resistance_cyl(p: Cyl1DParams) -> float:\n    """R_cond,cyl = ln(ro/ri)/(2π k L) [K/W]."""\n    validate_params_cyl(p)\n    return np.log(p.ro / p.ri) / (2.0 * np.pi * p.k * p.L)\n\n\n# ============================\n# Condução radial 1D — Esfera (regime permanente, sem geração)\n# ============================\n\n@dataclass(frozen=True)\nclass Sph1DParams:\n    k: float     # W/(m·K)\n    ri: float    # m\n    ro: float    # m\n    Ti: float    # K (em r=ri)\n    To: float    # K (em r=ro)\n\n\ndef validate_params_sph(p: Sph1DParams) -> None:\n    if p.k <= 0:\n        raise ValueError("k deve ser > 0 [W/(m·K)].")\n    if p.ri <= 0 or p.ro <= 0:\n        raise ValueError("ri e ro devem ser > 0 [m].")\n    if p.ro <= p.ri:\n        raise ValueError("Deve valer ro > ri.")\n    if p.Ti <= 0 or p.To <= 0:\n        raise ValueError("Temperaturas devem estar em Kelvin (T > 0 K).")\n\n\ndef solve_sphere_steady_1d(p: Sph1DParams, nr: int = 300) -> Tuple[np.ndarray, np.ndarray, float, np.ndarray]:\n    """\n    Casca esférica (1D radial), regime permanente, k const, sem geração.\n    BCs: T(ri)=Ti, T(ro)=To.\n\n    Retorna:\n      r [m], T(r) [K], Qdot [W], qpp(r) [W/m²] onde qpp(r)=Qdot/(4π r²)\n    """\n    validate_params_sph(p)\n    if nr < 20:\n        raise ValueError("nr deve ser >= 20.")\n\n    r = np.linspace(p.ri, p.ro, nr)\n\n    # Perfil de temperatura (forma 1/r)\n    denom = (1.0 / p.ro - 1.0 / p.ri)\n    T = p.Ti + (p.To - p.Ti) * ((1.0 / r - 1.0 / p.ri) / denom)\n\n    # Taxa de calor (constante)\n    Qdot = 4.0 * np.pi * p.k * (p.Ti - p.To) / (1.0 / p.ri - 1.0 / p.ro)\n\n    # Fluxo radial por área local\n    qpp = Qdot / (4.0 * np.pi * r**2)\n\n    return r, T, Qdot, qpp\n\n\ndef thermal_resistance_sph(p: Sph1DParams) -> float:\n    """R_cond,sph = (1/(4πk))*(1/ri - 1/ro) [K/W]."""\n    validate_params_sph(p)\n    return (1.0 / (4.0 * np.pi * p.k)) * (1.0 / p.ri - 1.0 / p.ro)\n\n\n# ============================\n# Cilindro 1D radial com convecção em r=ro (regime permanente, sem geração)\n# BCs: T(ri)=Ti e -k dT/dr|ro = h (T(ro)-Tinf)\n# ============================\n\nfrom dataclasses import dataclass\nfrom typing import Tuple\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass Cyl1DConvParams:\n    k: float     # W/(m·K)\n    ri: float    # m\n    ro: float    # m\n    L: float     # m (comprimento)\n    Ti: float    # K\n    h: float     # W/(m²·K)\n    Tinf: float  # K\n\n\ndef validate_params_cyl_conv(p: Cyl1DConvParams) -> None:\n    if p.k <= 0:\n        raise ValueError("k deve ser > 0 [W/(m·K)].")\n    if p.h <= 0:\n        raise ValueError("h deve ser > 0 [W/(m²·K)].")\n    if p.L <= 0:\n        raise ValueError("L deve ser > 0 [m].")\n    if p.ri <= 0 or p.ro <= 0:\n        raise ValueError("ri e ro devem ser > 0 [m].")\n    if p.ro <= p.ri:\n        raise ValueError("Deve valer ro > ri.")\n    if p.Ti <= 0 or p.Tinf <= 0:\n        raise ValueError("Temperaturas devem estar em Kelvin (T > 0 K).")\n\n\ndef thermal_resistance_cyl_conv(p: Cyl1DConvParams) -> Tuple[float, float, float]:\n    """\n    Retorna (R_cond, R_conv, R_tot) [K/W]\n    """\n    validate_params_cyl_conv(p)\n    R_cond = np.log(p.ro / p.ri) / (2.0 * np.pi * p.k * p.L)\n    R_conv = 1.0 / (p.h * 2.0 * np.pi * p.ro * p.L)\n    return R_cond, R_conv, R_cond + R_conv\n\n\ndef solve_cylinder_steady_1d_conv(p: Cyl1DConvParams, nr: int = 300) -> Tuple[np.ndarray, np.ndarray, float, np.ndarray, float]:\n    """\n    Casca cilíndrica (1D radial), regime permanente, sem geração.\n    BCs: T(ri)=Ti e convecção em r=ro.\n    Retorna:\n      r [m], T(r) [K], Qdot [W], qpp(r) [W/m²], Ts = T(ro) [K]\n    """\n    validate_params_cyl_conv(p)\n    if nr < 20:\n        raise ValueError("nr deve ser >= 20.")\n\n    r = np.linspace(p.ri, p.ro, nr)\n\n    R_cond, R_conv, R_tot = thermal_resistance_cyl_conv(p)\n\n    # Taxa de calor (constante)\n    Qdot = (p.Ti - p.Tinf) / R_tot\n\n    # Perfil de temperatura (a partir de Ti e do termo log)\n    T = p.Ti - (Qdot / (2.0 * np.pi * p.k * p.L)) * np.log(r / p.ri)\n\n    # Fluxo local por área\n    qpp = Qdot / (2.0 * np.pi * r * p.L)\n\n    Ts = T[-1]  # T(ro)\n    return r, T, Qdot, qpp, Ts\n\n\ndef biot_cylinder_external(p: Cyl1DConvParams) -> float:\n    """\n    Número de Biot com comprimento característico Lc = (ro-ri).\n    (Definição didática; não é a única possível.)\n    """\n    validate_params_cyl_conv(p)\n    Lc = (p.ro - p.ri)\n    return p.h * Lc / p.k\n\n\n# ============================\n# Meio semi-infinito — condução transiente 1D\n# Superfície subitamente imposta\n# ============================\n\nfrom dataclasses import dataclass\nfrom typing import Tuple\nimport numpy as np\nfrom math import erf, sqrt, pi\n\n\n@dataclass(frozen=True)\nclass SemiInfiniteParams:\n    k: float      # W/(m·K)\n    alpha: float  # m²/s\n    Ti: float     # K\n    Ts: float     # K\n\n\ndef validate_params_semi(p: SemiInfiniteParams) -> None:\n    if p.k <= 0:\n        raise ValueError("k deve ser > 0.")\n    if p.alpha <= 0:\n        raise ValueError("alpha deve ser > 0.")\n    if p.Ti <= 0 or p.Ts <= 0:\n        raise ValueError("Temperaturas devem estar em Kelvin.")\n\n\ndef temperature_semi_infinite(\n    x: np.ndarray, t: float, p: SemiInfiniteParams\n) -> np.ndarray:\n    """\n    T(x,t) para meio semi-infinito com Ts imposto em x=0.\n    """\n    validate_params_semi(p)\n    if t <= 0:\n        raise ValueError("t deve ser > 0.")\n\n    eta = x / (2.0 * np.sqrt(p.alpha * t))\n    T = p.Ts + (p.Ti - p.Ts) * np.array([erf(e) for e in eta])\n    return T\n\n\ndef surface_heat_flux(t: float, p: SemiInfiniteParams) -> float:\n    """\n    Fluxo de calor na superfície x=0.\n    """\n    validate_params_semi(p)\n    if t <= 0:\n        raise ValueError("t deve ser > 0.")\n\n    qpp = p.k * (p.Ti - p.Ts) / np.sqrt(pi * p.alpha * t)\n    return qpp\n'))
_write(os.path.join(ROOT, 'widgets_wall1d.py'), textwrap.dedent('# pubensino/ht/widgets_wall1d.py\n# UI mínima (ipywidgets) para Notebook 1\n\nimport ipywidgets as widgets\nfrom IPython.display import display\n\nfrom pubensino.ht.wall1d import Wall1DParams, solve_wall_steady_1d\nfrom pubensino.ht.viz_wall1d import plot_temperature_profile\n\n\ndef wall1d_interactive():\n    """\n    Cria sliders para (k, L, A, T1, T2) e plota T(x).\n    Interatividade mínima: atualizar quando o slider muda.\n    """\n\n    # Sliders (faixas conservadoras e coerentes)\n    w_k  = widgets.FloatLogSlider(value=15.0, base=10, min=-1, max=3, step=0.01,\n                                  description=\'k [W/mK]\', continuous_update=False)\n    w_L  = widgets.FloatSlider(value=0.02, min=0.001, max=0.20, step=0.001,\n                               description=\'L [m]\', continuous_update=False)\n    w_A  = widgets.FloatSlider(value=1.0, min=0.01, max=5.0, step=0.01,\n                               description=\'A [m²]\', continuous_update=False)\n\n    # Temperaturas em K\n    w_T1 = widgets.FloatSlider(value=350.0, min=250.0, max=600.0, step=1.0,\n                               description=\'T1 [K]\', continuous_update=False)\n    w_T2 = widgets.FloatSlider(value=300.0, min=250.0, max=600.0, step=1.0,\n                               description=\'T2 [K]\', continuous_update=False)\n\n    out = widgets.Output()\n\n    def update(_=None):\n        out.clear_output(wait=True)\n\n        p = Wall1DParams(\n            k=w_k.value,\n            L=w_L.value,\n            A=w_A.value,\n            T1=w_T1.value,\n            T2=w_T2.value\n        )\n\n        with out:\n            x, T, qpp, Qdot = solve_wall_steady_1d(p, nx=250)\n            plot_temperature_profile(x, T, p, qpp, Qdot)\n\n    for w in [w_k, w_L, w_A, w_T1, w_T2]:\n        w.observe(update, names="value")\n\n    controls = widgets.VBox([\n        widgets.HTML("<b>Parâmetros</b>"),\n        w_k, w_L, w_A,\n        widgets.HBox([w_T1, w_T2]),\n    ])\n\n    update()\n    display(widgets.VBox([controls, out]))\n\n\ndef wall1d_conv_interactive():\n    import ipywidgets as widgets\n    from IPython.display import display\n\n    from pubensino.ht.wall1d import Wall1DConvParams, solve_wall_steady_1d_conv\n    from pubensino.ht.viz_wall1d import plot_temperature_profile_conv\n\n    w_k  = widgets.FloatLogSlider(value=15.0, base=10, min=-1, max=3, step=0.01,\n                                  description=\'k [W/mK]\', continuous_update=False)\n    w_L  = widgets.FloatSlider(value=0.02, min=0.001, max=0.20, step=0.001,\n                               description=\'L [m]\', continuous_update=False)\n    w_A  = widgets.FloatSlider(value=1.0, min=0.01, max=5.0, step=0.01,\n                               description=\'A [m²]\', continuous_update=False)\n\n    w_h  = widgets.FloatLogSlider(value=100.0, base=10, min=0, max=5, step=0.01,\n                                  description=\'h [W/m²K]\', continuous_update=False)\n\n    w_T1   = widgets.FloatSlider(value=350.0, min=250.0, max=600.0, step=1.0,\n                                 description=\'T1 [K]\', continuous_update=False)\n    w_Tinf = widgets.FloatSlider(value=300.0, min=250.0, max=600.0, step=1.0,\n                                 description=\'T∞ [K]\', continuous_update=False)\n\n    out = widgets.Output()\n\n    def update(_=None):\n        out.clear_output(wait=True)\n        p = Wall1DConvParams(k=w_k.value, L=w_L.value, A=w_A.value, T1=w_T1.value, h=w_h.value, Tinf=w_Tinf.value)\n        with out:\n            x, T, qpp, Qdot, Ts = solve_wall_steady_1d_conv(p, nx=250)\n            plot_temperature_profile_conv(x, T, p, qpp, Qdot, Ts)\n\n    for w in [w_k, w_L, w_A, w_h, w_T1, w_Tinf]:\n        w.observe(update, names="value")\n\n    controls = widgets.VBox([\n        widgets.HTML("<b>Parâmetros (convecção em x=L)</b>"),\n        w_k, w_L, w_A, w_h, w_T1, w_Tinf,\n    ])\n\n    update()\n    display(widgets.VBox([controls, out]))\n\ndef cylinder_interactive():\n    import ipywidgets as widgets\n    from IPython.display import display\n\n    from pubensino.ht.wall1d import Cyl1DParams, solve_cylinder_steady_1d\n    from pubensino.ht.viz_wall1d import plot_cylinder_profiles\n\n    w_k  = widgets.FloatLogSlider(value=15.0, base=10, min=-1, max=3, step=0.01,\n                                  description=\'k [W/mK]\', continuous_update=False)\n    w_ri = widgets.FloatSlider(value=0.01, min=1e-3, max=0.20, step=1e-3,\n                               description=\'ri [m]\', continuous_update=False)\n    w_ro = widgets.FloatSlider(value=0.03, min=2e-3, max=0.30, step=1e-3,\n                               description=\'ro [m]\', continuous_update=False)\n    w_L  = widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05,\n                               description=\'L [m]\', continuous_update=False)\n\n    w_Ti = widgets.FloatSlider(value=350.0, min=250.0, max=700.0, step=1.0,\n                               description=\'Ti [K]\', continuous_update=False)\n    w_To = widgets.FloatSlider(value=300.0, min=250.0, max=700.0, step=1.0,\n                               description=\'To [K]\', continuous_update=False)\n\n    out = widgets.Output()\n\n    def update(_=None):\n        out.clear_output(wait=True)\n        # garantir ro > ri\n        ri = w_ri.value\n        ro = max(w_ro.value, ri + 1e-4)\n        if ro != w_ro.value:\n            w_ro.value = ro\n\n        p = Cyl1DParams(k=w_k.value, ri=ri, ro=ro, L=w_L.value, Ti=w_Ti.value, To=w_To.value)\n        with out:\n            r, T, Qdot, qpp = solve_cylinder_steady_1d(p, nr=350)\n            plot_cylinder_profiles(r, T, qpp, p, Qdot)\n\n    for w in [w_k, w_ri, w_ro, w_L, w_Ti, w_To]:\n        w.observe(update, names="value")\n\n    controls = widgets.VBox([\n        widgets.HTML("<b>Casca cilíndrica (1D radial, regime permanente)</b>"),\n        w_k, w_ri, w_ro, w_L, w_Ti, w_To,\n    ])\n\n    update()\n    display(widgets.VBox([controls, out]))\n\n\ndef sphere_interactive():\n    import ipywidgets as widgets\n    from IPython.display import display\n\n    from pubensino.ht.wall1d import Sph1DParams, solve_sphere_steady_1d\n    from pubensino.ht.viz_wall1d import plot_sphere_profiles\n\n    w_k  = widgets.FloatLogSlider(value=15.0, base=10, min=-1, max=3, step=0.01,\n                                  description=\'k [W/mK]\', continuous_update=False)\n    w_ri = widgets.FloatSlider(value=0.01, min=1e-3, max=0.20, step=1e-3,\n                               description=\'ri [m]\', continuous_update=False)\n    w_ro = widgets.FloatSlider(value=0.03, min=2e-3, max=0.30, step=1e-3,\n                               description=\'ro [m]\', continuous_update=False)\n\n    w_Ti = widgets.FloatSlider(value=350.0, min=250.0, max=700.0, step=1.0,\n                               description=\'Ti [K]\', continuous_update=False)\n    w_To = widgets.FloatSlider(value=300.0, min=250.0, max=700.0, step=1.0,\n                               description=\'To [K]\', continuous_update=False)\n\n    out = widgets.Output()\n\n    def update(_=None):\n        out.clear_output(wait=True)\n        ri = w_ri.value\n        ro = max(w_ro.value, ri + 1e-4)\n        if ro != w_ro.value:\n            w_ro.value = ro\n\n        p = Sph1DParams(k=w_k.value, ri=ri, ro=ro, Ti=w_Ti.value, To=w_To.value)\n        with out:\n            r, T, Qdot, qpp = solve_sphere_steady_1d(p, nr=350)\n            plot_sphere_profiles(r, T, qpp, p, Qdot)\n\n    for w in [w_k, w_ri, w_ro, w_Ti, w_To]:\n        w.observe(update, names="value")\n\n    controls = widgets.VBox([\n        widgets.HTML("<b>Casca esférica (1D radial, regime permanente)</b>"),\n        w_k, w_ri, w_ro, w_Ti, w_To,\n    ])\n\n    update()\n    display(widgets.VBox([controls, out]))\n\n\ndef cylinder_conv_interactive():\n    import ipywidgets as widgets\n    from IPython.display import display\n\n    from pubensino.ht.wall1d import (\n        Cyl1DConvParams,\n        solve_cylinder_steady_1d_conv,\n        thermal_resistance_cyl_conv,\n        biot_cylinder_external\n    )\n    from pubensino.ht.viz_wall1d import plot_cylinder_profiles_conv\n\n    w_k  = widgets.FloatLogSlider(value=15.0, base=10, min=-1, max=3, step=0.01,\n                                  description=\'k [W/mK]\', continuous_update=False)\n\n    w_ri = widgets.FloatSlider(value=0.01, min=1e-3, max=0.20, step=1e-3,\n                               description=\'ri [m]\', continuous_update=False)\n    w_ro = widgets.FloatSlider(value=0.03, min=2e-3, max=0.30, step=1e-3,\n                               description=\'ro [m]\', continuous_update=False)\n\n    w_L  = widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05,\n                               description=\'L [m]\', continuous_update=False)\n\n    w_h  = widgets.FloatLogSlider(value=100.0, base=10, min=0, max=5, step=0.01,\n                                  description=\'h [W/m²K]\', continuous_update=False)\n\n    w_Ti   = widgets.FloatSlider(value=350.0, min=250.0, max=800.0, step=1.0,\n                                 description=\'Ti [K]\', continuous_update=False)\n    w_Tinf = widgets.FloatSlider(value=300.0, min=250.0, max=800.0, step=1.0,\n                                 description=\'T∞ [K]\', continuous_update=False)\n\n    out = widgets.Output()\n\n    def update(_=None):\n        out.clear_output(wait=True)\n\n        ri = w_ri.value\n        ro = max(w_ro.value, ri + 1e-4)  # garante ro>ri\n        if ro != w_ro.value:\n            w_ro.value = ro\n\n        p = Cyl1DConvParams(k=w_k.value, ri=ri, ro=ro, L=w_L.value, Ti=w_Ti.value, h=w_h.value, Tinf=w_Tinf.value)\n\n        with out:\n            r, T, Qdot, qpp, Ts = solve_cylinder_steady_1d_conv(p, nr=350)\n            R_cond, R_conv, _ = thermal_resistance_cyl_conv(p)\n            Bi = biot_cylinder_external(p)\n            plot_cylinder_profiles_conv(r, T, qpp, p, Qdot, Ts, Bi, R_cond, R_conv)\n\n    for w in [w_k, w_ri, w_ro, w_L, w_h, w_Ti, w_Tinf]:\n        w.observe(update, names="value")\n\n    controls = widgets.VBox([\n        widgets.HTML("<b>Cilindro 1D (T(ri)=Ti; convecção em r=ro)</b>"),\n        w_k, w_ri, w_ro, w_L, w_h, w_Ti, w_Tinf,\n    ])\n\n    update()\n    display(widgets.VBox([controls, out]))\n\n\ndef semi_infinite_interactive():\n    import ipywidgets as widgets\n    from IPython.display import display\n    import numpy as np\n\n    from pubensino.ht.wall1d import (\n        SemiInfiniteParams,\n        temperature_semi_infinite,\n        surface_heat_flux\n    )\n    from pubensino.ht.viz_wall1d import plot_semi_infinite_profiles_time\n\n    w_k = widgets.FloatLogSlider(value=15.0, base=10, min=-1, max=3, step=0.01,\n                                 description=\'k [W/mK]\', continuous_update=False)\n\n    w_alpha = widgets.FloatLogSlider(value=1e-5, base=10, min=-7, max=-3, step=0.01,\n                                     description=\'α [m²/s]\', continuous_update=False)\n\n    w_Ti = widgets.FloatSlider(value=350.0, min=250.0, max=800.0, step=1.0,\n                               description=\'Ti [K]\', continuous_update=False)\n\n    w_Ts = widgets.FloatSlider(value=300.0, min=250.0, max=800.0, step=1.0,\n                               description=\'Ts [K]\', continuous_update=False)\n\n    w_t = widgets.FloatLogSlider(value=1.0, base=10, min=-2, max=4, step=0.01,\n                                 description=\'t_ref [s]\', continuous_update=False)\n    w_xmax = widgets.FloatLogSlider(\n    value=0.01, base=10, min=-4, max=0, step=0.01,\n    description=\'x_max [m]\', continuous_update=False\n)\n\n\n    # quantas curvas de tempo mostrar\n    w_span = widgets.Dropdown(\n        options=[("t/10, t, 10t", "decade"), ("t/4, t/2, t, 2t, 4t", "octave")],\n        value="decade",\n        description="tempos",\n    )\n\n    out = widgets.Output()\n\n    def update(_=None):\n        out.clear_output(wait=True)\n\n        p = SemiInfiniteParams(\n            k=w_k.value,\n            alpha=w_alpha.value,\n            Ti=w_Ti.value,\n            Ts=w_Ts.value\n        )\n\n        t_ref = float(w_t.value)\n\n        # Lista de tempos ao redor de t_ref\n        if w_span.value == "decade":\n            t_list = np.array([t_ref/10.0, t_ref, 10.0*t_ref], dtype=float)\n        else:\n            t_list = np.array([t_ref/4.0, t_ref/2.0, t_ref, 2.0*t_ref, 4.0*t_ref], dtype=float)\n\n        # Evitar t muito pequeno (por segurança numérica e física do modelo)\n        t_list = np.clip(t_list, 1e-9, None)\n\n        x_max = float(w_xmax.value)\n        x = np.linspace(0.0, x_max, 350)\n\n\n        # Computar perfis e fluxos\n        T_list = []\n        qpp_list = []\n        for tt in t_list:\n            T_list.append(temperature_semi_infinite(x, tt, p))\n            qpp_list.append(surface_heat_flux(tt, p))\n\n        with out:\n            plot_semi_infinite_profiles_time(x, T_list, t_list, p, qpp_list, t_ref)\n\n    for w in [w_k, w_alpha, w_Ti, w_Ts, w_t, w_xmax, w_span,w_xmax]:\n        w.observe(update, names="value")\n\n    controls = widgets.VBox([\n        widgets.HTML("<b>Meio semi-infinito — múltiplos tempos</b>"),\n        w_k, w_alpha, w_Ti, w_Ts, w_t, w_span\n    ])\n\n    update()\n    display(widgets.VBox([controls, out]))\n'))
_write(os.path.join(ROOT, 'rad_utils.py'), textwrap.dedent('import numpy as np\nimport matplotlib.pyplot as plt\n\nH = 6.62607015e-34\nC = 2.99792458e8\nKB = 1.380649e-23\n\nSIGMA = 5.670374419e-8\nWIEN_B = 2.897771955e-3\nC2 = H * C / KB\n\ndef E_lambda_blackbody(lam_m: np.ndarray, T: float) -> np.ndarray:\n    lam_m = np.asarray(lam_m, dtype=float)\n    if np.any(lam_m <= 0):\n        raise ValueError("lam_m deve ser positivo (m).")\n    if T <= 0:\n        raise ValueError("T deve ser positivo (K).")\n    a = 2.0 * np.pi * H * C**2\n    expo = C2 / (lam_m * T)\n    denom = np.expm1(expo)\n    return a / (lam_m**5 * denom)\n\ndef wien_peak_lambda(T: float) -> float:\n    if T <= 0:\n        raise ValueError("T deve ser positivo (K).")\n    return WIEN_B / T\n\ndef integrate_spectral_exitance(lam_m: np.ndarray, E_lam: np.ndarray) -> float:\n    lam_m = np.asarray(lam_m, dtype=float)\n    E_lam = np.asarray(E_lam, dtype=float)\n    if lam_m.ndim != 1 or E_lam.ndim != 1:\n        raise ValueError("lam_m e E_lam devem ser vetores 1D.")\n    if lam_m.size != E_lam.size:\n        raise ValueError("lam_m e E_lam devem ter o mesmo tamanho.")\n    if np.any(np.diff(lam_m) <= 0):\n        raise ValueError("lam_m deve ser estritamente crescente.")\n    return float(np.trapz(E_lam, lam_m))\n\ndef plot_blackbody_spectra(lam_um: np.ndarray, T_list, show_wien_peaks: bool = True):\n    lam_um = np.asarray(lam_um, dtype=float)\n    if np.any(lam_um <= 0):\n        raise ValueError("lam_um deve ser positivo (μm).")\n    lam_m = lam_um * 1e-6\n\n    plt.figure()\n    for T in T_list:\n        E = E_lambda_blackbody(lam_m, float(T))\n        plt.plot(lam_um, E, label=f"T = {T:.0f} K")\n        if show_wien_peaks:\n            lam_peak_m = wien_peak_lambda(float(T))\n            lam_peak_um = lam_peak_m * 1e6\n            if lam_um.min() <= lam_peak_um <= lam_um.max():\n                E_peak = E_lambda_blackbody(np.array([lam_peak_m]), float(T))[0]\n                plt.plot([lam_peak_um], [E_peak], marker="o")\n\n    plt.xlabel(r"Comprimento de onda, $\\lambda$ ($\\mu$m)")\n    plt.ylabel(r"$E_\\lambda^b(\\lambda,T)$ (W m$^{-3}$)")\n    plt.grid(True, which="both", alpha=0.3)\n    plt.legend()\n    plt.title("Espectro de corpo negro (Lei de Planck)")\n\ndef validate_stefan_boltzmann(T: float, lam_um_max: float = 200.0) -> dict:\n    if T <= 0:\n        raise ValueError("T deve ser positivo (K).")\n    lam_um = np.linspace(0.01, lam_um_max, 200000)\n    lam_m = lam_um * 1e-6\n    E = E_lambda_blackbody(lam_m, T)\n    Eb_num = integrate_spectral_exitance(lam_m, E)\n    Eb_SB = SIGMA * T**4\n    rel_err = (Eb_num - Eb_SB) / Eb_SB\n    return {"Eb_num": Eb_num, "Eb_SB": Eb_SB, "rel_err": rel_err}\n'))
_write(os.path.join(ROOT, 'viz_wall1d.py'), textwrap.dedent('# pubensino/ht/viz_wall1d.py\n# Rotinas de visualização (plot simples) para Notebook 1\n\nimport matplotlib.pyplot as plt\n\n\ndef plot_temperature_profile(x, T, p, qpp, Qdot) -> None:\n    """\n    Plot mínimo: T(x) em K.\n    Inclui painel numérico com q\'\' e Qdot.\n    """\n    fig, ax = plt.subplots()\n    ax.plot(x, T, linewidth=2.5, label="T(x)")\n\n    ax.set_xlabel("x [m]")\n    ax.set_ylabel("T [K]")\n    ax.set_title("Parede plana 1D — condução em regime permanente")\n    ax.grid(True, alpha=0.25)\n\n    # Painel numérico (mínimo)\n    txt = (\n        f"k = {p.k:.4g} W/(m·K)\\n"\n        f"L = {p.L:.4g} m\\n"\n        f"A = {p.A:.4g} m²\\n"\n        f"T1 = {p.T1:.4g} K,  T2 = {p.T2:.4g} K\\n"\n        f"q\'\' = {qpp:.4g} W/m²\\n"\n        f"Q̇ = {Qdot:.4g} W"\n    )\n    ax.text(\n        0.02, 0.02, txt, transform=ax.transAxes,\n        va="bottom", ha="left",\n        bbox=dict(boxstyle="round", alpha=0.10)\n    )\n\n    ax.legend(loc="best")\n    plt.tight_layout()\n    plt.show()\n\ndef plot_temperature_profile_conv(x, T, p, qpp, Qdot, Ts) -> None:\n    fig, ax = plt.subplots()\n    ax.plot(x, T, linewidth=2.5, label="T(x)")\n\n    # Marca Ts em x=L\n    ax.scatter([p.L], [Ts], zorder=3, label=r"$T_s=T(L)$")\n\n    # Linha de T_inf\n    ax.axhline(p.Tinf, linewidth=1.5, linestyle="--", label=r"$T_\\infty$")\n\n    ax.set_xlabel("x [m]")\n    ax.set_ylabel("T [K]")\n    ax.set_title("Parede 1D — T(0)=T1 e convecção em x=L")\n    ax.grid(True, alpha=0.25)\n\n    txt = (\n        f"k = {p.k:.4g} W/(m·K)\\n"\n        f"L = {p.L:.4g} m\\n"\n        f"A = {p.A:.4g} m²\\n"\n        f"h = {p.h:.4g} W/(m²·K)\\n"\n        f"T1 = {p.T1:.4g} K,  T∞ = {p.Tinf:.4g} K\\n"\n        f"Ts = {Ts:.4g} K\\n"\n        f"q\'\' = {qpp:.4g} W/m²\\n"\n        f"Q̇ = {Qdot:.4g} W"\n    )\n    ax.text(0.02, 0.02, txt, transform=ax.transAxes,\n            va="bottom", ha="left", bbox=dict(boxstyle="round", alpha=0.10))\n\n    ax.legend(loc="best")\n    plt.tight_layout()\n    plt.show()\n\n\ndef plot_cylinder_profiles(r, T, qpp, p, Qdot) -> None:\n    """Duas curvas na mesma figura: T(r) e q\'\'(r) (eixos separados)."""\n    fig, ax1 = plt.subplots()\n\n    ax1.plot(r, T, linewidth=2.5, label="T(r)")\n    ax1.set_xlabel("r [m]")\n    ax1.set_ylabel("T [K]")\n    ax1.set_title("Casca cilíndrica 1D — regime permanente")\n    ax1.grid(True, alpha=0.25)\n\n    # Segundo eixo para q\'\'(r)\n    ax2 = ax1.twinx()\n    ax2.plot(r, qpp, linewidth=2.0, linestyle="--", label="q\'\'(r)")\n    ax2.set_ylabel("q\'\'(r) [W/m²]")\n\n    txt = (\n        f"k = {p.k:.4g} W/(m·K)\\n"\n        f"ri = {p.ri:.4g} m, ro = {p.ro:.4g} m\\n"\n        f"L = {p.L:.4g} m\\n"\n        f"Ti = {p.Ti:.4g} K, To = {p.To:.4g} K\\n"\n        f"Q̇ = {Qdot:.4g} W"\n    )\n    ax1.text(0.02, 0.02, txt, transform=ax1.transAxes,\n             va="bottom", ha="left", bbox=dict(boxstyle="round", alpha=0.10))\n\n    # Legenda combinada\n    lines = ax1.get_lines() + ax2.get_lines()\n    labels = [ln.get_label() for ln in lines]\n    ax1.legend(lines, labels, loc="best")\n\n    plt.tight_layout()\n    plt.show()\n\n\ndef plot_sphere_profiles(r, T, qpp, p, Qdot) -> None:\n    """Duas curvas na mesma figura: T(r) e q\'\'(r) (eixos separados)."""\n    fig, ax1 = plt.subplots()\n\n    ax1.plot(r, T, linewidth=2.5, label="T(r)")\n    ax1.set_xlabel("r [m]")\n    ax1.set_ylabel("T [K]")\n    ax1.set_title("Casca esférica 1D — regime permanente")\n    ax1.grid(True, alpha=0.25)\n\n    ax2 = ax1.twinx()\n    ax2.plot(r, qpp, linewidth=2.0, linestyle="--", label="q\'\'(r)")\n    ax2.set_ylabel("q\'\'(r) [W/m²]")\n\n    txt = (\n        f"k = {p.k:.4g} W/(m·K)\\n"\n        f"ri = {p.ri:.4g} m, ro = {p.ro:.4g} m\\n"\n        f"Ti = {p.Ti:.4g} K, To = {p.To:.4g} K\\n"\n        f"Q̇ = {Qdot:.4g} W"\n    )\n    ax1.text(0.02, 0.02, txt, transform=ax1.transAxes,\n             va="bottom", ha="left", bbox=dict(boxstyle="round", alpha=0.10))\n\n    lines = ax1.get_lines() + ax2.get_lines()\n    labels = [ln.get_label() for ln in lines]\n    ax1.legend(lines, labels, loc="best")\n\n    plt.tight_layout()\n    plt.show()\n\n\ndef plot_cylinder_profiles_conv(r, T, qpp, p, Qdot, Ts, Bi, R_cond, R_conv) -> None:\n    """\n    Figura única com dois eixos:\n      - T(r) (eixo esquerdo)\n      - q\'\'(r) (eixo direito)\n    Inclui Ts=T(ro) e linha horizontal em Tinf.\n    """\n    fig, ax1 = plt.subplots()\n\n    ax1.plot(r, T, linewidth=2.5, label="T(r)")\n    ax1.scatter([p.ro], [Ts], zorder=3, label=r"$T_s=T(r_o)$")\n    ax1.axhline(p.Tinf, linewidth=1.5, linestyle="--", label=r"$T_\\infty$")\n\n    ax1.set_xlabel("r [m]")\n    ax1.set_ylabel("T [K]")\n    ax1.set_title("Cilindro 1D — T(ri)=Ti e convecção em r=ro")\n    ax1.grid(True, alpha=0.25)\n\n    ax2 = ax1.twinx()\n    ax2.plot(r, qpp, linewidth=2.0, linestyle="--", label="q\'\'(r)")\n    ax2.set_ylabel("q\'\'(r) [W/m²]")\n\n    txt = (\n        f"k = {p.k:.4g} W/(m·K)\\n"\n        f"h = {p.h:.4g} W/(m²·K)\\n"\n        f"ri = {p.ri:.4g} m, ro = {p.ro:.4g} m, L = {p.L:.4g} m\\n"\n        f"Ti = {p.Ti:.4g} K, T∞ = {p.Tinf:.4g} K\\n"\n        f"Ts = {Ts:.4g} K\\n"\n        f"Q̇ = {Qdot:.4g} W\\n"\n        f"Bi = {Bi:.4g}\\n"\n        f"Rcond = {R_cond:.4g} K/W\\n"\n        f"Rconv = {R_conv:.4g} K/W"\n    )\n    ax1.text(0.02, 0.02, txt, transform=ax1.transAxes,\n             va="bottom", ha="left", bbox=dict(boxstyle="round", alpha=0.10))\n\n    # legenda combinada\n    lines = ax1.get_lines() + ax2.get_lines()\n    labels = [ln.get_label() for ln in lines]\n    ax1.legend(lines, labels, loc="best")\n\n    plt.tight_layout()\n    plt.show()\n\n\ndef plot_semi_infinite_profiles_time(\n    x, T_list, t_list, p, qpp_list, t_ref\n) -> None:\n    """\n    Figura única:\n      - Curvas T(x,t) para vários tempos (t_list)\n      - Inset: q\'\'(0,t) vs t (para os mesmos tempos)\n    """\n    fig, ax = plt.subplots()\n\n    # Perfis T(x,t)\n    for T, tt in zip(T_list, t_list):\n        ax.plot(x, T, linewidth=2.0, label=f"t = {tt:.3g} s")\n\n    ax.set_xlabel("x [m]")\n    ax.set_ylabel("T [K]")\n    ax.set_title("Meio semi-infinito — condução transiente (múltiplos tempos)")\n    ax.grid(True, alpha=0.25)\n    ax.legend(loc="best")\n\n    # Painel numérico (referência do slider)\n    txt = (\n        f"k = {p.k:.4g} W/(m·K)\\n"\n        f"α = {p.alpha:.4g} m²/s\\n"\n        f"Ti = {p.Ti:.4g} K\\n"\n        f"Ts = {p.Ts:.4g} K\\n"\n        f"t_ref = {t_ref:.4g} s"\n    )\n    ax.text(\n        0.02, 0.02, txt, transform=ax.transAxes,\n        va="bottom", ha="left", bbox=dict(boxstyle="round", alpha=0.10)\n    )\n\n    # Inset: q\'\'(0,t) vs t (decai ~ t^{-1/2})\n    ax_in = ax.inset_axes([0.62, 0.12, 0.35, 0.30])  # [x0, y0, w, h]\n    ax_in.plot(t_list, qpp_list, marker="o", linewidth=1.5)\n    ax_in.set_xscale("log")\n    ax_in.set_yscale("log")\n    ax_in.set_xlabel("t [s]", fontsize=9)\n    ax_in.set_ylabel("q\'\'(0,t) [W/m²]", fontsize=9)\n    ax_in.grid(True, alpha=0.25)\n\n    plt.tight_layout()\n    plt.show()\n\n'))
_write(os.path.join(ROOT, 'viz_fins1d.py'), textwrap.dedent('# pubensino/ht/viz_fins1d.py\nimport matplotlib.pyplot as plt\n\ndef plot_profile(x, T, title):\n    plt.figure()\n    plt.plot(x, T)\n    plt.xlabel("x [m]")\n    plt.ylabel("T [K]")\n    plt.title(title)\n    plt.grid(True)\n    plt.show()\n\ndef plot_vs_Nf(Ns, Q_tot, eta_o):\n    plt.figure()\n    plt.plot(Ns, Q_tot)\n    plt.xlabel("N_f")\n    plt.ylabel("Q_total [W]")\n    plt.title("Q_total vs N_f")\n    plt.grid(True)\n    plt.show()\n\n    plt.figure()\n    plt.plot(Ns, eta_o)\n    plt.xlabel("N_f")\n    plt.ylabel("η_o [-]")\n    plt.title("η_o vs N_f")\n    plt.grid(True)\n    plt.show()\n'))
_write(os.path.join(ROOT, 'fins1d.py'), textwrap.dedent('# pubensino/ht/fins1d.py\nimport numpy as np\n\ndef geometry_rectangular(w, t):\n    """Aleta retangular (placa): Ac=wt, P=2(w+t)."""\n    Ac = w * t\n    P  = 2.0 * (w + t)\n    return Ac, P\n\ndef geometry_cylindrical(D):\n    """Pino cilíndrico: Ac=pi D^2/4, P=pi D."""\n    Ac = np.pi * D**2 / 4.0\n    P  = np.pi * D\n    return Ac, P\n\ndef fin_parameter_m(h, P, k, Ac):\n    """m = sqrt(h P / (k Ac))."""\n    return np.sqrt(h * P / (k * Ac))\n\ndef theta_profile(x, L, m, bc, theta_b, h=None, k=None, theta_L=None):\n    """\n    Solução analítica para theta(x)=T(x)-T_inf em aleta reta de seção constante.\n\n    bc:\n      - "adiabatic_tip": dtheta/dx(L)=0\n      - "convective_tip": -k Ac dtheta/dx|L = h Ac theta(L)\n      - "prescribed_tip": theta(L)=theta_L\n      - "infinite_fin": L->infinito\n    """\n    x = np.asarray(x, dtype=float)\n\n    if bc == "adiabatic_tip":\n        return theta_b * np.cosh(m * (L - x)) / np.cosh(m * L)\n\n    if bc == "convective_tip":\n        if (h is None) or (k is None):\n            raise ValueError("convective_tip requer h e k.")\n        beta = h / (k * m)\n        num = np.cosh(m * (L - x)) + beta * np.sinh(m * (L - x))\n        den = np.cosh(m * L)       + beta * np.sinh(m * L)\n        return theta_b * num / den\n\n    if bc == "prescribed_tip":\n        if theta_L is None:\n            raise ValueError("prescribed_tip requer theta_L.")\n        B = (theta_L - theta_b * np.cosh(m * L)) / np.sinh(m * L)\n        return theta_b * np.cosh(m * x) + B * np.sinh(m * x)\n\n    if bc == "infinite_fin":\n        return theta_b * np.exp(-m * x)\n\n    raise ValueError("BC inválida.")\n\ndef heat_rate_Qf(L, m, bc, theta_b, k, Ac, h=None, theta_L=None):\n    """Qf = -k Ac dtheta/dx|_{x=0}."""\n    if bc == "adiabatic_tip":\n        return k * Ac * m * theta_b * np.tanh(m * L)\n\n    if bc == "convective_tip":\n        if h is None:\n            raise ValueError("convective_tip requer h.")\n        beta = h / (k * m)\n        num  = np.sinh(m * L) + beta * np.cosh(m * L)\n        den  = np.cosh(m * L) + beta * np.sinh(m * L)\n        return k * Ac * m * theta_b * (num / den)\n\n    if bc == "prescribed_tip":\n        if theta_L is None:\n            raise ValueError("prescribed_tip requer theta_L.")\n        B = (theta_L - theta_b * np.cosh(m * L)) / np.sinh(m * L)\n        return -k * Ac * (m * B)\n\n    if bc == "infinite_fin":\n        return k * Ac * m * theta_b\n\n    raise ValueError("BC inválida.")\n\ndef fin_areas(P, L, Ac, include_tip=False):\n    """Af ~ P L (+Ac se incluir ponta). Abf = Ac."""\n    Af = P * L + (Ac if include_tip else 0.0)\n    Abf = Ac\n    return Af, Abf\n\ndef metrics(Qf, h, Af, Abf, theta_b):\n    """(eta_f, eps_f)."""\n    eta = Qf / (h * Af  * theta_b) if h * Af  * theta_b != 0 else np.nan\n    eps = Qf / (h * Abf * theta_b) if h * Abf * theta_b != 0 else np.nan\n    return eta, eps\n\ndef fin_array(Qf, h, Ab, Abf, Af, theta_b, Nf):\n    """\n    Modelo de conjunto (nível de sistema):\n      Q_total = Nf Qf + h (Ab - Nf Abf) theta_b\n      eta_o = Q_total / (h (Ab + Nf Af) theta_b)\n    Impõe: Ab - Nf Abf >= 0.\n    """\n    Nf_max = int(np.floor(Ab / Abf)) if Abf > 0 else 0\n    N_use  = min(int(Nf), max(Nf_max, 0))\n\n    A_exposed = Ab - N_use * Abf\n    Q_total   = N_use * Qf + h * A_exposed * theta_b\n\n    denom = h * (Ab + N_use * Af) * theta_b\n    eta_o = Q_total / denom if denom != 0 else np.nan\n\n    return Q_total, eta_o, N_use, Nf_max, A_exposed\n'))
_write(os.path.join(ROOT, 'widgets_fins1d.py'), textwrap.dedent('# pubensino/ht/widgets_fins1d.py\nimport numpy as np\nimport ipywidgets as widgets\nfrom IPython.display import display, clear_output\n\nfrom . import fins1d\nfrom . import viz_fins1d\n\ndef fin_single():\n    geom = widgets.Dropdown(\n        options=[("Retangular (w,t)", "rect"), ("Cilíndrica (D)", "cyl")],\n        value="rect", description="Geometria:"\n    )\n\n    w = widgets.FloatText(value=0.02, description="w [m]:")\n    t = widgets.FloatText(value=0.001, description="t [m]:")\n    D = widgets.FloatText(value=0.005, description="D [m]:")\n\n    k = widgets.FloatText(value=200.0, description="k [W/mK]:")\n    h = widgets.FloatText(value=50.0,  description="h [W/m²K]:")\n    L = widgets.FloatText(value=0.05,  description="L [m]:")\n\n    Tb   = widgets.FloatText(value=373.0, description="T_b [K]:")\n    Tinf = widgets.FloatText(value=293.0, description="T_inf [K]:")\n\n    bc = widgets.Dropdown(\n        options=[\n            ("Ponta adiabática", "adiabatic_tip"),\n            ("Ponta convectiva", "convective_tip"),\n            ("Ponta a T prescrita", "prescribed_tip"),\n            ("Aleta infinita", "infinite_fin"),\n        ],\n        value="adiabatic_tip", description="BC ponta:"\n    )\n\n    Ttip = widgets.FloatText(value=303.0, description="T_ponta [K]:")\n    include_tip = widgets.Checkbox(value=False, description="Incluir ponta em A_f")\n\n    out = widgets.Output()\n\n    def _toggle():\n        if geom.value == "rect":\n            w.layout.display, t.layout.display, D.layout.display = "", "", "none"\n        else:\n            w.layout.display, t.layout.display, D.layout.display = "none", "none", ""\n\n        Ttip.layout.display = "" if bc.value == "prescribed_tip" else "none"\n\n    def _update(*args):\n        with out:\n            clear_output(wait=True)\n            _toggle()\n\n            if geom.value == "rect":\n                Ac, P = fins1d.geometry_rectangular(w.value, t.value)\n                geom_str = "Retangular"\n            else:\n                Ac, P = fins1d.geometry_cylindrical(D.value)\n                geom_str = "Cilíndrica"\n\n            theta_b = Tb.value - Tinf.value\n            m = fins1d.fin_parameter_m(h.value, P, k.value, Ac)\n\n            if bc.value == "infinite_fin":\n                L_plot = max(L.value, 5.0 / max(m, 1e-12))\n            else:\n                L_plot = L.value\n            x = np.linspace(0.0, L_plot, 400)\n\n            theta_L = (Ttip.value - Tinf.value) if bc.value == "prescribed_tip" else None\n\n            theta = fins1d.theta_profile(x, L.value, m, bc.value, theta_b, h=h.value, k=k.value, theta_L=theta_L)\n            Qf = fins1d.heat_rate_Qf(L.value, m, bc.value, theta_b, k=k.value, Ac=Ac, h=h.value, theta_L=theta_L)\n\n            Af, Abf = fins1d.fin_areas(P, L.value, Ac, include_tip=include_tip.value)\n            eta, eps = fins1d.metrics(Qf, h.value, Af, Abf, theta_b)\n\n            print("=== Aleta individual ===")\n            print(f"Geometria: {geom_str}")\n            print(f"A_c = {Ac:.6e} m² | P = {P:.6e} m")\n            print(f"m = {m:.6e} 1/m | mL = {(m*L.value):.3f}")\n            print(f"Q_f = {Qf:.6f} W")\n            print(f"η_f = {eta:.6f} | ε_f = {eps:.6f}")\n\n            T = theta + Tinf.value\n            viz_fins1d.plot_profile(x, T, f"T(x) — {bc.label}")\n\n    for wid in [geom, w, t, D, k, h, L, Tb, Tinf, bc, Ttip, include_tip]:\n        wid.observe(_update, "value")\n\n    _update()\n\n    uiL = widgets.VBox([geom, w, t, D, k, h, L])\n    uiR = widgets.VBox([Tb, Tinf, bc, Ttip, include_tip])\n    display(widgets.HBox([uiL, uiR]), out)\n\ndef fin_array():\n    # conjunto típico: aletas retangulares\n    w = widgets.FloatText(value=0.02,  description="w [m]:")\n    t = widgets.FloatText(value=0.001, description="t [m]:")\n\n    L = widgets.FloatText(value=0.05,  description="L [m]:")\n    k = widgets.FloatText(value=200.0, description="k [W/mK]:")\n    h = widgets.FloatText(value=50.0,  description="h [W/m²K]:")\n\n    Tb   = widgets.FloatText(value=373.0, description="T_b [K]:")\n    Tinf = widgets.FloatText(value=293.0, description="T_inf [K]:")\n\n    bc = widgets.Dropdown(\n        options=[\n            ("Ponta adiabática", "adiabatic_tip"),\n            ("Ponta convectiva", "convective_tip"),\n            ("Aleta infinita", "infinite_fin"),\n        ],\n        value="adiabatic_tip", description="BC ponta:"\n    )\n\n    Ab = widgets.FloatText(value=0.01, description="A_base [m²]:")\n    Nf = widgets.IntSlider(value=10, min=1, max=300, step=1, description="N_f:")\n\n    include_tip = widgets.Checkbox(value=False, description="Incluir ponta em A_f")\n    out = widgets.Output()\n\n    def _update(*args):\n        with out:\n            clear_output(wait=True)\n\n            Ac, P = fins1d.geometry_rectangular(w.value, t.value)\n            theta_b = Tb.value - Tinf.value\n            m = fins1d.fin_parameter_m(h.value, P, k.value, Ac)\n\n            Qf = fins1d.heat_rate_Qf(L.value, m, bc.value, theta_b, k=k.value, Ac=Ac, h=h.value)\n\n            Af, Abf = fins1d.fin_areas(P, L.value, Ac, include_tip=include_tip.value)\n\n            Q_total, eta_o, N_use, N_max, A_exp = fins1d.fin_array(Qf, h.value, Ab.value, Abf, Af, theta_b, Nf.value)\n\n            print("=== Conjunto de aletas ===")\n            print(f"A_base = {Ab.value:.6e} m² | A_bf = {Abf:.6e} m²")\n            print(f"N_f solicitado = {Nf.value} | N_f usado = {N_use} | N_f,max = {N_max}")\n            print(f"Área exposta = {A_exp:.6e} m²")\n            print(f"Q_f (1 aleta) = {Qf:.6f} W")\n            print(f"Q_total = {Q_total:.6f} W | η_o = {eta_o:.6f}")\n\n            if N_max >= 1:\n                Ns = np.arange(1, N_max + 1)\n                Qs = np.zeros_like(Ns, dtype=float)\n                et = np.zeros_like(Ns, dtype=float)\n                for i, n in enumerate(Ns):\n                    Qs[i], et[i], *_ = fins1d.fin_array(Qf, h.value, Ab.value, Abf, Af, theta_b, int(n))\n                viz_fins1d.plot_vs_Nf(Ns, Qs, et)\n\n    for wid in [w, t, L, k, h, Tb, Tinf, bc, Ab, Nf, include_tip]:\n        wid.observe(_update, "value")\n\n    _update()\n    display(widgets.VBox([\n        widgets.HBox([w, t, L]),\n        widgets.HBox([k, h]),\n        widgets.HBox([Tb, Tinf]),\n        widgets.HBox([bc, include_tip]),\n        widgets.HBox([Ab, Nf]),\n    ]), out)\n'))
_write(os.path.join(ROOT, 'rad_exchange.py'), textwrap.dedent('# pubensino/ht/rad_exchange.py\n"""Rotinas para radiação térmica (superfícies difusas-cinzentas)."""\n\nfrom __future__ import annotations\nimport numpy as np\n\nSIGMA = 5.670374419e-8  # Stefan–Boltzmann (W/m²/K⁴)\n\ndef linearized_hr(eps: float, Tm: float) -> float:\n    """h_r = 4 ε σ Tm³."""\n    eps = float(eps); Tm = float(Tm)\n    if not (0.0 < eps <= 1.0):\n        raise ValueError("eps deve estar em (0,1].")\n    if Tm <= 0:\n        raise ValueError("Tm deve ser positivo (K).")\n    return 4.0 * eps * SIGMA * Tm**3\n\ndef net_to_large_enclosure(A: float, eps: float, Ts, Tsur: float):\n    """q = ε σ A (Ts⁴ - Tsur⁴). Aceita Ts escalar ou array."""\n    A = float(A); eps = float(eps)\n    Ts = np.asarray(Ts, dtype=float); Tsur = float(Tsur)\n    if A <= 0:\n        raise ValueError("A deve ser positiva.")\n    if not (0.0 < eps <= 1.0):\n        raise ValueError("eps deve estar em (0,1].")\n    if np.any(Ts <= 0) or Tsur <= 0:\n        raise ValueError("Temperaturas devem ser positivas (K).")\n    return eps * SIGMA * A * (Ts**4 - Tsur**4)\n\ndef effective_emissivity_two_surfaces_unitF(eps1: float, eps2: float) -> float:\n    """ε_eff = 1 / (1/ε1 + 1/ε2 - 1)."""\n    eps1=float(eps1); eps2=float(eps2)\n    if not (0.0 < eps1 <= 1.0) or not (0.0 < eps2 <= 1.0):\n        raise ValueError("eps1 e eps2 devem estar em (0,1].")\n    denom = (1.0/eps1) + (1.0/eps2) - 1.0\n    return 1.0/denom\n\ndef net_between_two_surfaces_unitF(A: float, eps1: float, eps2: float, T1: float, T2: float) -> float:\n    """q = σ (T1⁴ - T2⁴) / ( (1-ε1)/(Aε1) + 1/A + (1-ε2)/(Aε2) )."""\n    A=float(A); eps1=float(eps1); eps2=float(eps2); T1=float(T1); T2=float(T2)\n    if A <= 0:\n        raise ValueError("A deve ser positiva.")\n    if not (0.0 < eps1 <= 1.0) or not (0.0 < eps2 <= 1.0):\n        raise ValueError("eps1 e eps2 devem estar em (0,1].")\n    if T1 <= 0 or T2 <= 0:\n        raise ValueError("Temperaturas devem ser positivas (K).")\n    R1 = (1.0 - eps1) / (A * eps1)\n    Rspace = 1.0 / A\n    R2 = (1.0 - eps2) / (A * eps2)\n    return SIGMA * (T1**4 - T2**4) / (R1 + Rspace + R2)\n'))
_write(os.path.join(ROOT, 'view_factors_mc.py'), textwrap.dedent('# pubensino/ht/view_factors_mc.py\n"""Estimativa Monte Carlo de fatores de forma entre retângulos paralelos."""\n\nfrom __future__ import annotations\nimport numpy as np\n\ndef sample_rect(n: int, Lx: float, Ly: float, cx: float=0.0, cy: float=0.0, rng=None):\n    rng = np.random.default_rng() if rng is None else rng\n    x = (rng.random(n) - 0.5) * Lx + cx\n    y = (rng.random(n) - 0.5) * Ly + cy\n    return x, y\n\ndef view_factor_parallel_rectangles_mc(\n    n: int, Lx1: float, Ly1: float, Lx2: float, Ly2: float, H: float,\n    dx: float = 0.0, dy: float = 0.0, seed: int | None = 0\n) -> dict:\n    if n <= 0: raise ValueError("n deve ser positivo.")\n    for val, name in [(Lx1,\'Lx1\'),(Ly1,\'Ly1\'),(Lx2,\'Lx2\'),(Ly2,\'Ly2\')]:\n        if val <= 0: raise ValueError(f"{name} deve ser positivo.")\n    if H <= 0: raise ValueError("H deve ser positivo.")\n\n    rng = np.random.default_rng(seed)\n    x1, y1 = sample_rect(n, Lx1, Ly1, 0.0, 0.0, rng)\n    x2, y2 = sample_rect(n, Lx2, Ly2, dx,  dy,  rng)\n\n    Rx = x2 - x1; Ry = y2 - y1\n    R2 = Rx*Rx + Ry*Ry + H*H\n    integrand = (H*H) / (np.pi * (R2**2))  # H^2 / (pi R^4)\n\n    A2 = Lx2 * Ly2\n    F12 = A2 * float(np.mean(integrand))\n    sigma_mean = float(np.std(integrand, ddof=1) / np.sqrt(n)) if n > 1 else float(\'nan\')\n    return {"F12": F12, "sigma_F12": A2*sigma_mean, "A1": Lx1*Ly1, "A2": A2}\n'))
_write(os.path.join(ROOT, 'widgets_radiation.py'), textwrap.dedent('# pubensino/ht/widgets_radiation.py\nfrom __future__ import annotations\n\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport ipywidgets as widgets\nfrom IPython.display import display, clear_output\n\nfrom . import rad_utils\nfrom . import rad_exchange\nfrom . import view_factors_mc\n\ndef blackbody_spectrum():\n    T = widgets.IntSlider(value=800, min=200, max=3000, step=50, description="T [K]:")\n    lam_max = widgets.FloatSlider(value=30.0, min=5.0, max=200.0, step=1.0, description="λ_max [μm]:")\n    npts = widgets.IntSlider(value=4000, min=500, max=20000, step=500, description="N_pts:")\n    out = widgets.Output()\n\n    def _update(*args):\n        with out:\n            clear_output(wait=True)\n            lam_um = np.linspace(0.05, lam_max.value, npts.value)\n            rad_utils.plot_blackbody_spectra(lam_um, [float(T.value)], show_wien_peaks=True)\n            lam_peak = rad_utils.wien_peak_lambda(float(T.value))*1e6\n            Eb_SB = rad_exchange.SIGMA * float(T.value)**4\n            print(f"λ_pico (Wien) ≈ {lam_peak:.3f} μm")\n            print(f"E_b = σT^4 ≈ {Eb_SB:.3e} W/m²")\n            plt.show()\n\n    for w in [T, lam_max, npts]:\n        w.observe(_update, "value")\n    _update()\n    display(widgets.VBox([widgets.HBox([T, lam_max, npts]), out]))\n\ndef grey_enclosure():\n    A = widgets.FloatText(value=0.10, description="A [m²]:")\n    eps = widgets.FloatSlider(value=0.8, min=0.05, max=1.0, step=0.01, description="ε:")\n    Ts = widgets.FloatSlider(value=600.0, min=250.0, max=2000.0, step=10.0, description="T_s [K]:")\n    Tsur = widgets.FloatSlider(value=300.0, min=250.0, max=1200.0, step=10.0, description="T_sur [K]:")\n    out = widgets.Output()\n\n    def _update(*args):\n        with out:\n            clear_output(wait=True)\n            q = float(rad_exchange.net_to_large_enclosure(A.value, eps.value, Ts.value, Tsur.value))\n            Tm = 0.5*(Ts.value + Tsur.value)\n            hr = rad_exchange.linearized_hr(eps.value, Tm)\n            q_lin = hr * A.value * (Ts.value - Tsur.value)\n            print("=== Superfície ↔ Envoltória grande (F=1) ===")\n            print(f"q_rad = {q:.3f} W")\n            print(f"h_r = {hr:.3f} W/m²K | q_lin = {q_lin:.3f} W")\n\n            Ts_vec = np.linspace(max(1.0, Ts.value*0.6), Ts.value*1.4, 200)\n            q_vec = rad_exchange.net_to_large_enclosure(A.value, eps.value, Ts_vec, Tsur.value)\n            plt.figure()\n            plt.plot(Ts_vec, q_vec)\n            plt.xlabel(r"$T_s$ [K]"); plt.ylabel(r"$q_{rad}$ [W]")\n            plt.grid(True, alpha=0.3)\n            plt.title("Não-linearidade de $q_{rad}(T_s)$")\n            plt.show()\n\n    for w in [A, eps, Ts, Tsur]:\n        w.observe(_update, "value")\n    _update()\n    display(widgets.VBox([widgets.HBox([A, eps]), widgets.HBox([Ts, Tsur]), out]))\n\ndef two_surfaces_unitF():\n    A = widgets.FloatText(value=0.10, description="A [m²]:")\n    eps1 = widgets.FloatSlider(value=0.8, min=0.05, max=1.0, step=0.01, description="ε1:")\n    eps2 = widgets.FloatSlider(value=0.6, min=0.05, max=1.0, step=0.01, description="ε2:")\n    T1 = widgets.FloatSlider(value=800.0, min=250.0, max=2000.0, step=10.0, description="T1 [K]:")\n    T2 = widgets.FloatSlider(value=300.0, min=250.0, max=2000.0, step=10.0, description="T2 [K]:")\n    out = widgets.Output()\n\n    def _update(*args):\n        with out:\n            clear_output(wait=True)\n            eps_eff = rad_exchange.effective_emissivity_two_surfaces_unitF(eps1.value, eps2.value)\n            q = rad_exchange.net_between_two_surfaces_unitF(A.value, eps1.value, eps2.value, T1.value, T2.value)\n            print("=== Duas superfícies, F12=1 ===")\n            print(f"ε_eff = {eps_eff:.4f} | q = {q:.3f} W")\n\n            eps2_vec = np.linspace(0.05, 1.0, 200)\n            q_vec = np.array([rad_exchange.net_between_two_surfaces_unitF(A.value, eps1.value, e2, T1.value, T2.value) for e2 in eps2_vec])\n            plt.figure()\n            plt.plot(eps2_vec, q_vec)\n            plt.xlabel(r"$\\epsilon_2$"); plt.ylabel(r"$q$ [W]")\n            plt.grid(True, alpha=0.3)\n            plt.title("Efeito de $\\epsilon_2$")\n            plt.show()\n\n    for w in [A, eps1, eps2, T1, T2]:\n        w.observe(_update, "value")\n    _update()\n    display(widgets.VBox([widgets.HBox([A, eps1, eps2]), widgets.HBox([T1, T2]), out]))\n\ndef view_factor_parallel_rectangles():\n    Lx1 = widgets.FloatText(value=0.20, description="Lx1 [m]:")\n    Ly1 = widgets.FloatText(value=0.20, description="Ly1 [m]:")\n    Lx2 = widgets.FloatText(value=0.20, description="Lx2 [m]:")\n    Ly2 = widgets.FloatText(value=0.20, description="Ly2 [m]:")\n    H  = widgets.FloatText(value=0.10, description="H [m]:")\n    dx = widgets.FloatText(value=0.00, description="dx [m]:")\n    dy = widgets.FloatText(value=0.00, description="dy [m]:")\n    n = widgets.IntSlider(value=20000, min=2000, max=200000, step=2000, description="N MC:")\n    out = widgets.Output()\n\n    def _update(*args):\n        with out:\n            clear_output(wait=True)\n            res12 = view_factors_mc.view_factor_parallel_rectangles_mc(int(n.value), Lx1.value, Ly1.value, Lx2.value, Ly2.value, H.value, dx.value, dy.value, seed=0)\n            res21 = view_factors_mc.view_factor_parallel_rectangles_mc(int(n.value), Lx2.value, Ly2.value, Lx1.value, Ly1.value, H.value, -dx.value, -dy.value, seed=1)\n            A1, A2 = res12["A1"], res12["A2"]\n            recip_err = abs(A1*res12["F12"] - A2*res21["F12"])\n            print("=== Fator de forma (MC) — retângulos paralelos ===")\n            print(f"F12 ≈ {res12[\'F12\']:.5f} (± {res12[\'sigma_F12\']:.5f}, 1σ)")\n            print(f"F21 ≈ {res21[\'F12\']:.5f} | |A1F12-A2F21| ≈ {recip_err:.2e}")\n\n            plt.figure()\n            x1 = np.array([-Lx1.value/2, Lx1.value/2, Lx1.value/2, -Lx1.value/2, -Lx1.value/2])\n            y1 = np.array([-Ly1.value/2, -Ly1.value/2, Ly1.value/2, Ly1.value/2, -Ly1.value/2])\n            x2 = np.array([-Lx2.value/2, Lx2.value/2, Lx2.value/2, -Lx2.value/2, -Lx2.value/2]) + dx.value\n            y2 = np.array([-Ly2.value/2, -Ly2.value/2, Ly2.value/2, Ly2.value/2, -Ly2.value/2]) + dy.value\n            plt.plot(x1, y1, label="Retângulo 1"); plt.plot(x2, y2, label="Retângulo 2")\n            plt.axis("equal"); plt.grid(True, alpha=0.3)\n            plt.xlabel("x [m]"); plt.ylabel("y [m]"); plt.legend()\n            plt.title("Projeção no plano xy")\n            plt.show()\n\n    for w in [Lx1, Ly1, Lx2, Ly2, H, dx, dy, n]:\n        w.observe(_update, "value")\n    _update()\n    display(widgets.VBox([widgets.HBox([Lx1, Ly1, Lx2, Ly2]), widgets.HBox([H, dx, dy, n]), out]))\n'))

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# teste rápido de import
import pubensino.ht

In [ ]:
from IPython.display import HTML
HTML("""
<style>
div.input {display:none;}
</style>
""")

# Notebook 1 — Condução de calor 1D

## Objetivos
Investigar a condução de calor em uma dimensão por meio de **soluções analíticas** clássicas, enfatizando a interpretação física e o papel das condições de contorno. A interatividade permite isolar o efeito de parâmetros térmicos e geométricos.

## Organização
O notebook está dividido em casos independentes:
1. Parede plana, **temperaturas prescritas** em $x=0$ e $x=L$.
2. Parede plana, **convecção** em $x=L$ (condição de terceira espécie).
3. Condução radial em **casca cilíndrica** (temperaturas prescritas).
4. Condução radial em **casca esférica** (temperaturas prescritas).
5. Casca cilíndrica com **convecção externa**.
6. Meio **semi-infinito**: condução transiente 1D.

## Variáveis e unidades
- Temperatura: $T$ [K]
- Condutividade térmica: $k$ [W/(m·K)]
- Difusividade térmica: $\alpha$ [m$^2$/s]
- Espessura (plano): $L$ [m]
- Área: $A$ [m$^2$]
- Coeficiente convectivo: $h$ [W/(m$^2$·K)]
- Taxa de calor: $\dot{Q}$ [W], fluxo: $q''$ [W/m$^2$]


## Hipóteses físicas e modelo matemático (caso plano: $T(0)=T_1$, $T(L)=T_2$)

**Enunciado físico.** Considera-se uma parede plana de espessura $L$, submetida a temperaturas prescritas $T_1$ e $T_2$ nas faces $x=0$ e $x=L$.

**Hipóteses.**
- Condução unidimensional (variação apenas em $x$).
- Regime permanente.
- Propriedades constantes ($k=$ const.).
- Sem geração volumétrica de calor.

**Lei de Fourier.**
$$
q''_x=-k\,\frac{dT}{dx},
$$
onde $q''_x$ é o fluxo de calor na direção $x$ [W/m$^2$].

**Equação governante (permanente, sem geração).**
$$
\frac{d^2T}{dx^2}=0.
$$

**Condições de contorno.**
$$
T(0)=T_1,\qquad T(L)=T_2.
$$

**Solução analítica e interpretação.**
$$
T(x)=T_1+(T_2-T_1)\frac{x}{L},
\qquad
q''_x=-k\,\frac{T_2-T_1}{L}.
$$
O perfil de temperatura é linear e o fluxo é constante no domínio, refletindo conservação de energia em regime permanente.

## Metodologia de modelagem
Não se resolve numericamente a EDP. As figuras são obtidas por **expressões analíticas paramétricas**, permitindo explorar de forma controlada:
- como $k$ e $L$ governam o gradiente $dT/dx$;
- como as condições de contorno alteram a resistência térmica equivalente.


In [ ]:
import importlib
import pubensino.ht.wall1d as wall1d
import pubensino.ht.widgets_wall1d as widgets_wall1d

importlib.reload(wall1d)
importlib.reload(widgets_wall1d)

widgets_wall1d.wall1d_interactive()

## Caso 2 — Parede plana com convecção em $x=L$

**Enunciado físico.** Mantém-se $T(0)=T_1$, mas substitui-se a condição $T(L)=T_2$ por uma condição convectiva com um fluido a $T_\infty$ e coeficiente $h$. A temperatura de superfície $T_s=T(L)$ é incógnita.

**Condição convectiva (3ª espécie).**
$$
-k\frac{dT}{dx}\Big|_{x=L}=h\,[T(L)-T_\infty].
$$

**Interpretação via resistências térmicas.**
A taxa de calor é determinada por duas resistências em série:
$$
R_{\text{cond}}=\frac{L}{kA},\qquad
R_{\text{conv}}=\frac{1}{hA},
\qquad
\dot Q=\frac{T_1-T_\infty}{R_{\text{cond}}+R_{\text{conv}}}.
$$
Quando $R_{\text{conv}}\gg R_{\text{cond}}$, a convecção limita a transferência; quando $R_{\text{cond}}\gg R_{\text{conv}}$, a condução domina.


In [ ]:
import importlib
import pubensino.ht.widgets_wall1d as widgets_wall1d
importlib.reload(widgets_wall1d)

widgets_wall1d.wall1d_conv_interactive()


## Caso 3 — Condução radial em casca cilíndrica (1D)

**Enunciado físico.** Condução em regime permanente através de uma casca cilíndrica longa, entre $r=r_i$ e $r=r_o$, com temperaturas prescritas $T(r_i)=T_i$ e $T(r_o)=T_o$.

**Equação governante (sem geração).**
$$
\frac{1}{r}\frac{d}{dr}\left(r\frac{dT}{dr}\right)=0.
$$

**Solução analítica.**
$$
T(r)=T_i + (T_o-T_i)\frac{\ln(r/r_i)}{\ln(r_o/r_i)}.
$$

**Taxa de calor e variação do fluxo com o raio.**
A taxa $\dot Q$ é constante ao longo do raio, mas o fluxo local depende da área cilíndrica:
$$
q_r''(r)=\frac{\dot Q}{2\pi r L}\propto \frac{1}{r}.
$$
Logo, mesmo com $\dot Q$ constante, $q_r''$ decai com $r$ por efeito geométrico.


In [ ]:
import importlib
import pubensino.ht.wall1d as wall1d
import pubensino.ht.widgets_wall1d as widgets_wall1d
import pubensino.ht.viz_wall1d as viz_wall1d

importlib.reload(wall1d)
importlib.reload(viz_wall1d)
importlib.reload(widgets_wall1d)
widgets_wall1d.cylinder_interactive()


## Condução radial em casca esférica (1D)

**Equação governante (permanente, sem geração).**
$$
\frac{1}{r^2}\frac{d}{dr}\left(r^2\frac{dT}{dr}\right)=0.
$$

**Solução com $T(r_i)=T_i$ e $T(r_o)=T_o$.**
$$
T(r)=T_i + (T_o-T_i)\,
\frac{\left(\frac{1}{r}-\frac{1}{r_i}\right)}
{\left(\frac{1}{r_o}-\frac{1}{r_i}\right)}.
$$

**Taxa de calor e fluxo local.**
$$
\dot Q=\frac{4\pi k (T_i-T_o)}{\left(\frac{1}{r_i}-\frac{1}{r_o}\right)},
\qquad
q_r''(r)=\frac{\dot Q}{4\pi r^2}\propto \frac{1}{r^2}.
$$
A redução mais rápida do fluxo ($\propto 1/r^2$) decorre do crescimento da área esférica $4\pi r^2$.


In [ ]:
import importlib
import pubensino.ht.wall1d as wall1d
import pubensino.ht.viz_wall1d as viz_wall1d
import pubensino.ht.widgets_wall1d as widgets_wall1d

importlib.reload(wall1d)
importlib.reload(viz_wall1d)
importlib.reload(widgets_wall1d)

widgets_wall1d.sphere_interactive()


## Caso cilíndrico — Convecção na superfície externa $r=r_o$

**Condições de contorno.**
$$
T(r_i)=T_i,\qquad
-k\frac{dT}{dr}\Big|_{r_o}=h\,[T(r_o)-T_\infty].
$$

**Interpretação via resistências térmicas.**
$$
R_{\text{cond}}=\frac{\ln(r_o/r_i)}{2\pi k L},
\qquad
R_{\text{conv}}=\frac{1}{h\,2\pi r_o L},
\qquad
\dot Q=\frac{T_i-T_\infty}{R_{\text{cond}}+R_{\text{conv}}}.
$$
Essa forma evidencia o papel competitivo entre a resistência radial (logarítmica) e a resistência convectiva na superfície externa.


In [ ]:
import importlib
import pubensino.ht.wall1d as wall1d
import pubensino.ht.viz_wall1d as viz_wall1d
import pubensino.ht.widgets_wall1d as widgets_wall1d

importlib.reload(wall1d)
importlib.reload(viz_wall1d)
importlib.reload(widgets_wall1d)

widgets_wall1d.cylinder_conv_interactive()


## Meio semi-infinito — condução transiente 1D

Um sólido pode ser tratado como **semi-infinito** quando a profundidade de penetração térmica
$\delta \sim \sqrt{\alpha t}$ é pequena em comparação com a espessura física do corpo no intervalo de tempo analisado.

**Problema canônico.** Superfície em $x=0$ subitamente imposta a $T_s$ (ou mantida), com interior inicialmente a $T_i$.

**Solução analítica (função erro).**
$$
T(x,t)=T_s+(T_i-T_s)\,
\operatorname{erf}\!\left(\frac{x}{2\sqrt{\alpha t}}\right).
$$

**Fluxo na superfície.**
$$
q''(0,t)=\frac{k\,(T_i-T_s)}{\sqrt{\pi\,\alpha\,t}}.
$$
A singularidade $q''\propto t^{-1/2}$ em tempos muito pequenos é consequência do gradiente inicial extremamente elevado, e deve ser interpretada dentro da idealização de condição de contorno “súbita”.


In [ ]:
import importlib
import pubensino.ht.widgets_wall1d as widgets_wall1d
importlib.reload(widgets_wall1d)

widgets_wall1d.semi_infinite_interactive()
